# Learning a quadratic pseudo-metric from distance measurements

Recall that pseudo-metric is a generalization of a metric space in which the distance between two distinct points can be zero.
We are given a set of $N$ pairs of points in $\mathbf{R}^n$, $x_1, \ldots, x_N$, and $y_1, \ldots, y_N$, together with a set of distances $d_1, \ldots, d_N > 0$.
The goal is to find (or estimate or learn) a quadratic pseudo-metric $d$

$$d(x,y) =  \left( (x-y)^T P(x-y) \right)^{1/2},$$

$P\in \mathbf{S}^n_{+}$, which approximates the given distances, i.e., $d(x_i, y_i) \approx d_i$.
(The pseudo-metric $d$ is a metric only when $P \succ 0$; when $P\succeq 0$ is singular, it is a pseudo-metric.)
  
To do this, we will choose $P\in \mathbf{S}^n_+$ that minimizes the mean squared error objective

$$f(P)=\frac{1}{N}\sum_{i=1}^N (d_i - d(x_i,y_i))^2.$$

### Theoretical part.
1. Show that the objective function $f$ is convex (Hint: expand the square and see what happens.)
2. Show that the convex program $\text{minimize }f(P)$, $P\succeq 0$ can be expressed by an equivalent conic program with linear objective and a number of conic constraints using the $R^n_+$ (nonnegative orthant cone), $Q^n$ (second order cone), $Q_r^n$ (rotated second order cone), $S^n_+$ (positive semidefinite cone).

### Programming Part
1. Solve the program $\text{minimize }f(P)$, $P\succeq 0$, preferably using a modelling package like ``cvxpy``. Note that "under the hood" your modelling package translates the program to the conic form in point 2. above.
2. Use the obtained $P$ to measure the mean square error for the test data ``X_test``, ``Y_test``, ``d_test``.
  
---- 
*This exercise originates from "Additional Exercises" collection for Convex Optimization textbook of S. Boyd and L. Vandenberghe. Used under permission*

In [1]:
import cvxpy as cp
import numpy as np
from scipy import linalg as la

In [2]:
# In this box we generate the input data

np.random.seed(5680)

n = 5 # Dimension
N = 100 # Number of samples

P_0 = np.random.randn(n,n)
P_0 = P_0.dot(P_0.T) + np.identity(n)
sqrtP_0 = la.sqrtm(P_0)

x = np.random.randn(N,n)
y = np.random.randn(N,n)

d = np.linalg.norm(sqrtP_0.dot((x-y).T),axis=0)    # distances according to metric P_0
d = np.maximum(d+np.random.randn(N),0)           # add random noise

N_test = 10 # Samples for test set
X_test = np.random.randn(N_test,n)
Y_test = np.random.randn(N_test,n)
d_test = np.linalg.norm(sqrtP_0.dot((X_test-Y_test).T),axis=0)  # distances according to metric P_0
d_test = np.maximum(d_test+np.random.randn(N_test),0)         # add random noise


## Theoretical part

Let $z_i = x_i - y_i$ and $P \in \mathbf{S}^n_+$ be the unknown matrix.
Then $d(x_i, y_i) = \sqrt{z_i^T P z_i}$ and

$$
(d_i - d(x_i, y_i))^2 = d_i^2 - 2 d_i \sqrt{z_i^T P z_i} + z_i^T P z_i.
$$

### Convexity of $f$

For each sample $i$, we have a sum of three terms:

- $d_i^2$ is a constant.
- $z_i^T P z_i = \operatorname{tr}(P \, z_i z_i^T)$ is affine (hence convex) in $P$.
- $\sqrt{z_i^T P z_i}$ is the concave function $\sqrt{\cdot}$ precomposed with the affine map $P \mapsto z_i^T P z_i$ (the argument of the square root is nonnegative on $\mathbf{S}^n_+$, so we are ok), hence concave in $P$. Since $d_i \ge 0$, the term $-2 d_i \sqrt{z_i^T P z_i}$ is convex in $P$.

A nonnegative sum of convex functions is convex, so $f$ is convex on $\mathbf{S}^n_+$.

### Conic reformulation

Let $t_i = \sqrt{z_i^T P z_i}$ and $s_i = z_i^T P z_i$. Then minimizing $f$ over $P \succeq 0$ is the same as

$$
\begin{aligned}
\text{minimize}\quad & \tilde f(s, t, P) = \frac{1}{N}\sum_{i=1}^N (d_i^2 - 2 d_i t_i + s_i) \\
\text{subject to}\quad
& P \succeq 0, t \succeq 0, s \succeq 0 \\
& t_i^2 = s_i, \qquad i = 1, \dots, N, \\
& s_i = z_i^T P z_i, \qquad i = 1, \dots, N.
\end{aligned}
$$

Replacing $t_i^2 = s_i$ by $t_i^2 \le s_i$ does not change the optimal value:
since $d_i \ge 0$, decreasing $t_i$ can only increase the objective, so at the optimum $t_i = \sqrt{s_i}$ anyway.

Using $s_i = \frac{(s_i+1)^2}{4} - \frac{(s_i-1)^2}{4}$,

$$
t_i^2 \le s_i
\iff
t_i^2 + (\frac{s_i-1}{2})^2 \le (\frac{s_i+1}{2})^2 .
$$

Both sides are nonnegative and we have $s \succeq 0$, so we can take square roots and equivalently write:

$$
||(t_i,\ \frac{s_i-1}{2})||_2 \le \frac{s_i+1}{2}
\iff
(t_i, \frac{s_i-1}{2}, \frac{s_i+1}{2}) \succeq_{Q^3} 0.
$$

Hence, the optimisation problem can be written as:

$$
\begin{aligned}
\text{minimize}\quad & \frac{1}{N}\sum_{i=1}^N (d_i^2 - 2 d_i t_i + s_i) \\
\text{subject to}\quad
& P \succeq 0, t \succeq 0, \\
& (t_i, \frac{s_i-1}{2}, \frac{s_i+1}{2}) \succeq_{Q^3} 0, \qquad i = 1, \dots, N, \\
& s_i = z_i^T P z_i, \qquad i = 1, \dots, N.
\end{aligned}
$$

The objective is linear, and all constraints are affine maps combined with generalized inequalities (with cones $\mathbf{S}^n_+$, $\mathbb{R}^N_+$ and $Q^3$), so this is a conic program.

### Equivalent reformulation using rotated cone

The rotated second-order cone is

$$
Q_r^n = \{(x, y, z) \in \mathbb{R}^{n-2} \times \mathbb{R} \times \mathbb{R} : \ ||x||_2^2 \le 2yz,\ y \ge 0,\ z \ge 0\}.
$$

It is the second-order cone $Q^n$ rotated by $45^\circ$ in the $(y, z)$ plane: since $(\frac{y+z}{\sqrt2})^2 - (\frac{y-z}{\sqrt2})^2 = 2yz$,

$$
(x, y, z) \succeq_{Q_r^n} 0 \iff (x, \frac{y-z}{\sqrt2}, \frac{y+z}{\sqrt2}) \succeq_{Q^n} 0 .
$$

Its use is that a product $yz$ appears on the right-hand side instead of a square, so a constraint of the form $x^2 \le yz$ needs no completing of the square. In our case take $x = t_i$, $y = s_i$, $z = \frac{1}{2}$. Then $2yz = s_i$ and

$$
t_i^2 \le s_i \iff (t_i, s_i, \frac{1}{2}) \succeq_{Q_r^3} 0 .
$$

The conic program is therefore the same as above with the $Q^3$ constraint replaced by

$$
(t_i, s_i, \frac{1}{2}) \succeq_{Q_r^3} 0, \qquad i = 1, \dots, N .
$$

The choice of $(y, z)$ is not unique: scaling $y$ by $\lambda > 0$ and $z$ by $\frac{1}{\lambda}$ leaves $2yz$ unchanged. With $y = \frac{s_i}{\sqrt2}$, $z = \frac{1}{\sqrt2}$ the rotation above gives $(t_i, \frac{s_i-1}{2}, \frac{s_i+1}{2}) \succeq_{Q^3} 0$, which is the $Q^3$ constraint obtained earlier by completing the square.

## Programming part

### Solve $\min f(P)$ subject to $P \succeq 0$ with `cvxpy`

We model the conic reformulation above. The objective
$\tfrac{1}{N}\sum_i (z_i^T P z_i - 2 d_i t_i + d_i^2)$ is linear in the
variables $(P, t)$; the constraint $t_i^2 \le z_i^T P z_i$ is recognized by
cvxpy as a rotated second-order cone constraint.

In [3]:
Z = x - y  # shape (N, n);  Z[i] = x_i - y_i

P = cp.Variable((n, n), symmetric=True)
t = cp.Variable(N, nonneg=True)

# quad_i = z_i^T P z_i, expressed as a linear function of P via trace(z z^T P)
quad = cp.hstack([cp.trace(np.outer(Z[i], Z[i]) @ P) for i in range(N)])

constraints = [P >> 0, cp.square(t) <= quad]

objective = cp.Minimize(cp.sum(quad - 2 * cp.multiply(d, t) + d ** 2) / N)

prob = cp.Problem(objective, constraints)
prob.solve()

P_hat = P.value
print("solver status:", prob.status)
print("training MSE f(P_hat) =", prob.value)
print("recovered P_hat =\n", P_hat)

solver status: optimal
training MSE f(P_hat) = 0.7130855075441379
recovered P_hat =
 [[ 5.84482483  3.06726069 -0.46025962  1.4322332   2.66300428]
 [ 3.06726069 12.41185509  1.3304393   4.8128188  -2.38449173]
 [-0.46025962  1.3304393   3.14600965 -0.60222919 -2.56565131]
 [ 1.4322332   4.8128188  -0.60222919  2.89191775  0.76923583]
 [ 2.66300428 -2.38449173 -2.56565131  0.76923583 10.55414405]]


### Test MSE

Using the learned $\hat P$, evaluate
$\mathrm{MSE}_{\text{test}} = \frac{1}{N_{\text{test}}}\sum_i \big(d_{\text{test},i} - \sqrt{(X_{\text{test},i}-Y_{\text{test},i})^T \hat P (X_{\text{test},i}-Y_{\text{test},i})}\big)^2.$

The test distances $d_{\text{test}}$ are noisy, like the training ones. This simulates the practical setting, where only noisy measurements are available and the true distances are never observed. Consequently even the true $P_0$ does not reach zero test MSE; its value (printed below as a sanity check) is the noise floor to compare $\hat P$ against.

In [4]:
Z_test = X_test - Y_test
d_hat_test = np.sqrt(np.einsum("ij,jk,ik->i", Z_test, P_hat, Z_test))
test_mse = np.mean((d_test - d_hat_test) ** 2)
print("test MSE =", test_mse)

# For reference, the MSE of the *true* generating P on the same data:
d_hat_true = np.sqrt(np.einsum("ij,jk,ik->i", Z_test, P_0, Z_test))
print("test MSE with true P (sanity check) =",
      np.mean((d_test - d_hat_true) ** 2))

test MSE = 0.9132385006215002
test MSE with true P (sanity check) = 0.7802951274920111


### Recovery of the ground truth $P_0$

The noisy test MSE cannot go below the noise floor. Since this is a simulation and $P_0$ is known, we can also check directly how well $\hat P$ recovers it: the relative Frobenius error of $\hat P$, and the MSE of the learned metric against the noiseless test distances. Neither is available in practice.

In [5]:
# Relative error of the recovered matrix
print("||P_hat - P_0||_F / ||P_0||_F =", np.linalg.norm(P_hat - P_0) / np.linalg.norm(P_0))

# MSE of the learned metric against the *noiseless* test distances (d_hat_true from the cell above)
print("test MSE vs noiseless distances =", np.mean((d_hat_true - d_hat_test) ** 2))

||P_hat - P_0||_F / ||P_0||_F = 0.09050792373759818
test MSE vs noiseless distances = 0.08953639040046339
